# Fine-tunning on Figure 3D MethylBERT data with EpigenDnabert2

This tutorial demonstrates:
1. **How to perform fine-tunning on the prepared data and save results to the local directory**.
---

### Step 0: Import needed classes and set paths

In [1]:
from methyldl.modelling.dnabert2 import EpigenDnabert2, TrainingArguments
from methyldl.data.dataset import SupervisedDataset
# data_path = "../Data/Curated/MethylBERT_Figure3D"
# data_path = "../Data/Curated/MethylBERT_Figure3D_customsplit"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr2"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr1_cpg_counts_selected"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40/chr8_dmr_cuts"
# data_path = "../Data/Curated/rrms_dmrs_only_mincov40_corrected_imporvement_experiments/chr19_top3000_global_dmrs_dmrs_cuts"
data_path = "../Data/Curated/experiment_c/chr19"
from torch import nn

x:\KULeuven-Masters\Master Thesis\methyldl\methyldl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Inspecting data in path 

### Step 1: Perform fine-tunning

In [ ]:
model_instance = EpigenDnabert2(use_cpg_methylation=True, max_sequence_length=1000, trust_remote_code=True)

training_args = TrainingArguments(
            run_name = "dnabert2_"+data_path.split("/")[-1],
            per_device_train_batch_size = 42,
            per_device_eval_batch_size = 100,
            gradient_accumulation_steps = 1,
            learning_rate = 3e-5,
            fp16 = True,
            save_steps = 50,
            output_dir ="output/dnabert2_experiment_c_"+data_path.split("/")[-1],
            eval_strategy = "steps",
            eval_steps = 50, 
            warmup_steps = 100, 
            logging_steps = 100, 
            num_train_epochs = 250, 
            overwrite_output_dir = True, 
            log_level = "info",
            find_unused_parameters = False,
            batch_eval_metrics = False,
            eval_and_save_results = True,
            remove_unused_columns=False,
            eval_accumulation_steps = 8,
            torch_empty_cache_steps = 10,
            prediction_loss_only=False,
            gradient_checkpointing=False,
            skip_memory_metrics=True,
            auto_find_batch_size=False,
            )

In [ ]:
model_instance.model.classifier = nn.Linear(768,out_features=3,bias=True)
model_instance.num_labels=3
model_instance.model.num_labels = 3
model_instance.model.config.problem_type = "single_label_classification"

In [ ]:
import os

In [ ]:
train_dataset = SupervisedDataset(tokenizer=model_instance.tokenizer, 
                                        data_path_or_list=os.path.join(data_path, "train"), 
                                        kmer=-1,data_interface="pandas")

val_dataset = SupervisedDataset(tokenizer=model_instance.tokenizer, 
                                        data_path_or_list=os.path.join(data_path, "valid"), 
                                        kmer=-1,data_interface="pandas")

In [ ]:
model_instance.fine_tune(train_dataset=train_dataset, 
                         val_dataset=val_dataset,
                         test_dataset="empty",
                         training_args=training_args, data_interface="pandas")